# 🏥 TriageAI — SFT + RL Training
**Model:** Qwen2.5-3B-Instruct | **Method:** Expert SFT → GRPO RL | **Team:** Block Dragon

In [ ]:
!pip install -q unsloth trl requests matplotlib datasets

In [ ]:
import os, json, random, time, requests, torch
import matplotlib.pyplot as plt
from datasets import Dataset

ENV_URL = "https://hinex-07-triage-ai-env.hf.space"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./triage_ai_trained"
os.makedirs(OUTPUT_DIR, exist_ok=True)

r = requests.get(f"{ENV_URL}/health", timeout=30)
print(f"Environment: {r.json()}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=4096, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
)
print(f"Loaded {MODEL_NAME}, trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
SYSTEM_PROMPT = """You are an expert Emergency Room triage physician AI. You manage a busy ER with limited beds, doctors, and one operating room.

GOAL: Save as many patients as possible by triaging, assigning resources, and treating efficiently.

ACTIONS (respond with EXACTLY one JSON object, nothing else):
1. {"action_type": "triage", "patient_id": "P001"}
2. {"action_type": "assign_bed", "patient_id": "P001"}
3. {"action_type": "assign_doctor", "patient_id": "P001", "params": {"doctor_id": 0}}
4. {"action_type": "order_treatment", "patient_id": "P001", "params": {"treatment": "medication"}}
5. {"action_type": "send_to_or", "patient_id": "P001"}
6. {"action_type": "discharge", "patient_id": "P001"}
7. {"action_type": "reassess", "patient_id": "P001"}
8. {"action_type": "submit"}

RULES:
- Triage critical patients FIRST (low SpO2, high HR, altered consciousness)
- Assign beds to urgent patients before stable ones
- Discharge non-urgent patients to free beds for critical ones
- Use OR for surgical emergencies only
- Respond with ONLY the JSON action object"""

def env_reset(task_id="task_easy", seed=None):
    seed = seed or random.randint(0,100000)
    return requests.post(f"{ENV_URL}/reset", json={"task_id":task_id,"seed":seed}, timeout=30).json()

def env_step(action):
    return requests.post(f"{ENV_URL}/step", json=action, timeout=30).json()

def format_obs(obs_data):
    obs = obs_data.get("observation", obs_data)
    lines = [f"=== ER (Step {obs.get('current_step',0)}/{obs.get('max_steps',0)}) ==="]
    lines.append(f"Action result: {obs.get('last_action_message','')}")
    s = obs.get("summary",{})
    lines.append(f"Patients: {s.get('total_patients',0)} total | {s.get('alive',0)} alive | {s.get('dead',0)} dead | {s.get('treated',0)} treated")
    h = obs.get("hospital",{})
    beds = h.get("beds",[])
    lines.append(f"Beds: {sum(1 for b in beds if not b.get('occupied'))}/{len(beds)} free")
    docs = h.get("doctors",[])
    lines.append(f"Doctors: {sum(1 for d in docs if not d.get('busy'))}/{len(docs)} available")
    or_i = h.get("or",{})
    lines.append(f"OR: {'Available' if or_i.get('available') else f'In use (cooldown {or_i.get(\"cooldown_steps\",0)})'}")
    for p in obs.get("waiting_patients",[]):
        v = p.get("vitals",{})
        tl = f" [Triage: {p['triage_level']}]" if p.get("triage_level") else ""
        lines.append(f"[WAIT] {p['id']} {p.get('name','')}: {p.get('chief_complaint','')} | HR={v.get('hr')} SpO2={v.get('spo2')}%{tl}")
        lines.append(f"  Symptoms: {p.get('symptoms','')}")
    for p in obs.get("admitted_patients",[]):
        v = p.get("vitals",{})
        tl = f" [Triage: {p.get('triage_level','')}]" if p.get("triage_level") else ""
        notes = f" | Notes: {p['clinical_notes']}" if p.get("clinical_notes") else ""
        lines.append(f"[BED] {p['id']} {p.get('name','')}: {p.get('chief_complaint','')} | HR={v.get('hr')} SpO2={v.get('spo2')}%{tl}{notes}")
    return "\n".join(lines)

def parse_action(text):
    text = text.strip()
    if '```' in text: text = text.split('```')[1] if len(text.split('```'))>1 else text
    try: return json.loads(text[text.index('{'):text.rindex('}')+1])
    except: return {"action_type":"submit"}

print("Helpers loaded!")

## Step 1: Generate Expert Training Data

In [ ]:
def get_sev(p):
    v = p.get('vitals',{})
    hr,spo2,con = v.get('hr',80),v.get('spo2',98),v.get('consciousness','alert')
    if con=='unresponsive': return 1
    if con=='confused': return 2
    if spo2<85: return 1
    if spo2<92: return 2
    if hr>130: return 1
    if hr>110: return 2
    return 5

def needs_surg(p):
    t = f"{p.get('clinical_notes','')} {p.get('chief_complaint','')} {p.get('symptoms','')}".lower()
    return any(k in t for k in ['laceration','fracture','hemorrhage','hematoma','dissection','torsion','obstruction','appendicitis','surgery','surgical','bleeding','arterial','immediate intervention','visible muscle'])

def expert_action(obs_data):
    obs = obs_data.get('observation',obs_data)
    w,a = obs.get('waiting_patients',[]),obs.get('admitted_patients',[])
    h = obs.get('hospital',{})
    fb = sum(1 for b in h.get('beds',[]) if not b.get('occupied'))
    fd = sum(1 for d in h.get('doctors',[]) if not d.get('busy'))
    or_ok = h.get('or',{}).get('available',False)
    # 1. Triage untriaged
    ut = [p for p in w if not p.get('triage_level')]
    if ut:
        ut.sort(key=get_sev)
        return {'action_type':'triage','patient_id':ut[0]['id']}
    # 2. Discharge stable admitted to free beds
    if fb == 0 and w:
        for p in sorted(a, key=lambda x: -(x.get('triage_level',5) or 5)):
            if (p.get('triage_level') or 5) >= 4:
                return {'action_type':'discharge','patient_id':p['id']}
    # 3. Assign beds to critical waiting
    if fb > 0 and w:
        sw = sorted(w, key=lambda p: p.get('triage_level',5))
        return {'action_type':'assign_bed','patient_id':sw[0]['id']}
    # 4. Assign doctors
    if fd > 0:
        for p in sorted(a, key=lambda x: x.get('triage_level',5) or 5):
            if not p.get('examined') and not p.get('clinical_notes'):
                return {'action_type':'assign_doctor','patient_id':p['id'],'params':{}}
    # 5. Treat examined
    for p in a:
        if p.get('clinical_notes') or p.get('examined'):
            if needs_surg(p) and or_ok:
                return {'action_type':'send_to_or','patient_id':p['id']}
            return {'action_type':'order_treatment','patient_id':p['id'],'params':{'treatment':'medication'}}
    # 6. Discharge remaining stable
    for p in a:
        if (p.get('triage_level') or 5) >= 4:
            return {'action_type':'discharge','patient_id':p['id']}
    for p in w:
        if (p.get('triage_level') or 5) >= 4:
            return {'action_type':'discharge','patient_id':p['id']}
    return {'action_type':'submit'}

print('Expert agent ready!')

In [ ]:
# Generate expert demonstrations
sft_data = []
expert_scores, expert_survivals = [], []

# DRY RUN: use 5/5/3. FULL RUN: use 40/40/20
tasks = [('task_easy',40),('task_medium',40),('task_hard',20)]

for task_id, n_eps in tasks:
    print(f'\n--- {task_id} ({n_eps} episodes) ---')
    for ep in range(n_eps):
        try:
            obs = env_reset(task_id, random.randint(0,100000))
            done, step = False, 0
            while not done and step < 40:
                obs_text = format_obs(obs)
                action = expert_action(obs)
                sft_data.append({'messages':[
                    {'role':'system','content':SYSTEM_PROMPT},
                    {'role':'user','content':obs_text},
                    {'role':'assistant','content':json.dumps(action)}
                ]})
                obs = env_step(action)
                done = obs.get('done',False)
                step += 1
            meta = obs.get('observation',{}).get('metadata',{})
            expert_scores.append(meta.get('composite_score',0))
            expert_survivals.append(meta.get('survival_rate',0))
            if (ep+1)%10==0:
                print(f'  Ep {ep+1}: examples={len(sft_data)} avg_score={sum(expert_scores[-10:])/len(expert_scores[-10:]):.3f} avg_surv={sum(expert_survivals[-10:])/len(expert_survivals[-10:]):.3f}')
        except Exception as e:
            print(f'  Error: {e}')
            time.sleep(1)

print(f'\nTotal SFT examples: {len(sft_data)}')
print(f'Expert avg score: {sum(expert_scores)/len(expert_scores):.3f}')
print(f'Expert avg survival: {sum(expert_survivals)/len(expert_survivals):.3f}')

## Step 2: Baseline Evaluation (Before Training)

In [ ]:
def run_eval(model, tokenizer, task_id='task_easy', n=3):
    scores, survs = [], []
    FastLanguageModel.for_inference(model)
    for ep in range(n):
        obs = env_reset(task_id, seed=ep*100)
        done, steps = False, 0
        while not done and steps < 20:
            obs_text = format_obs(obs)
            msgs = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':obs_text}]
            inp = tokenizer(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=200, temperature=0.1, do_sample=True)
            resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
            obs = env_step(parse_action(resp))
            done = obs.get('done',False)
            steps += 1
        m = obs.get('observation',{}).get('metadata',{})
        scores.append(m.get('composite_score',0))
        survs.append(m.get('survival_rate',0))
    return sum(scores)/len(scores), sum(survs)/len(survs)

print('=== BASELINE (Before Training) ===')
baseline = {}
for t in ['task_easy','task_medium','task_hard']:
    s,v = run_eval(model, tokenizer, t, 3)
    baseline[t] = (s,v)
    print(f'  {t}: score={s:.3f} survival={v:.3f}')

## Step 3: SFT Training on Expert Data

In [ ]:
from trl import SFTTrainer, SFTConfig

# Convert to HF Dataset
def format_for_sft(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(sft_data)
ds = ds.map(format_for_sft)
print(f'SFT dataset: {len(ds)} examples')

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        save_steps=100,
        max_seq_length=4096,
        dataset_text_field='text',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

print('Starting SFT training...')
results = trainer.train()
print(f'Training done! Loss: {results.training_loss:.4f}')

## Step 4: Post-Training Evaluation

In [ ]:
print('=== AFTER SFT TRAINING ===')
after = {}
for t in ['task_easy','task_medium','task_hard']:
    s,v = run_eval(model, tokenizer, t, 3)
    after[t] = (s,v)
    print(f'  {t}: score={s:.3f} survival={v:.3f}')

print('\n=== IMPROVEMENT ===')
for t in ['task_easy','task_medium','task_hard']:
    bs,bv = baseline[t]
    a_s,av = after[t]
    print(f'  {t}: score {bs:.3f} -> {a_s:.3f} (+{a_s-bs:.3f}) | survival {bv:.3f} -> {av:.3f} (+{av-bv:.3f})')

## Step 5: Training Curves & Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before/After bar chart
tasks = ['task_easy','task_medium','task_hard']
x = range(len(tasks))
w = 0.35

before_scores = [baseline[t][0] for t in tasks]
after_scores = [after[t][0] for t in tasks]
before_surv = [baseline[t][1] for t in tasks]
after_surv = [after[t][1] for t in tasks]

axes[0].bar([i-w/2 for i in x], before_scores, w, label='Before', color='#e74c3c', alpha=0.8)
axes[0].bar([i+w/2 for i in x], after_scores, w, label='After SFT', color='#2ecc71', alpha=0.8)
axes[0].set_title('Composite Score', fontsize=13, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(['Easy','Medium','Hard'])
axes[0].set_ylim(0,1); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].bar([i-w/2 for i in x], before_surv, w, label='Before', color='#e74c3c', alpha=0.8)
axes[1].bar([i+w/2 for i in x], after_surv, w, label='After SFT', color='#2ecc71', alpha=0.8)
axes[1].set_title('Survival Rate', fontsize=13, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(['Easy','Medium','Hard'])
axes[1].set_ylim(0,1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('TriageAI: Before vs After SFT Training', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
print('\n' + '='*70)
print(f"{'Task':<15} {'Metric':<12} {'Before':>10} {'After':>10} {'Change':>10}")
print('='*70)
for t in ['task_easy','task_medium','task_hard']:
    bs,bv = baseline[t]
    a_s,av = after[t]
    print(f"{t:<15} {'Score':<12} {bs:>10.3f} {a_s:>10.3f} {a_s-bs:>+10.3f}")
    print(f"{'':<15} {'Survival':<12} {bv:>10.3f} {av:>10.3f} {av-bv:>+10.3f}")
    print('-'*70)
print('='*70)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')